In [1]:
import pandas as pd
from datetime import datetime
import numpy as np
#customer
customers = pd.DataFrame([
    [1,"Ali","ali@mail.com","Baku","2023-01-01"],
    [2,"Nigar","nigar@mail.com","Ganja","2023-02-10"],
    [3,"Murad","murad@mail.com","Sumgait","2023-03-15"],
    [4,"Aysel","aysel@mail.com","Baku","2023-04-20"],
    [5,"Rashad","rashad@mail.com","Baku","2023-05-05"],
    [6,"Leyla","leyla@mail.com","Ganja","2023-06-01"],
    [7,"Tural","tural@mail.com","Sumgait","2023-06-10"],
    [8,"Sabina","sabina@mail.com","Baku","2023-07-01"]
], columns=["customer_id","name","email","city","signup_date"])

# Movies
movies = pd.DataFrame([
    [1,"Inception","Sci-Fi",2010,5],
    [2,"Titanic","Romance",1997,5],
    [3,"Avengers","Action",2012,4],
    [4,"Joker","Drama",2019,5],
    [5,"Frozen","Animation",2013,4],
    [6,"Matrix","Sci-Fi",1999,5],
    [7,"Batman","Action",2008,4],
    [8,"Interstellar","Sci-Fi",2014,5],
    [9,"Shrek","Animation",2001,3],
    [10,"Gladiator","Action",2000,5]
], columns=["movie_id","title","genre","release_year","rating"])

In [2]:
np.random.seed(0)

rentals= pd.DataFrame({
    "rental_id" : range(1,26),
    "customer_id": np.random.randint(1,9,25),
    "movie_id": np.random.randint(1,11,25),
    "rental_date": pd.date_range("2023-08-01", periods=25),
    "return_date": pd.date_range("2023-08-05", periods=25),
    "price": np.random.randint(5,20,25)
})

In [3]:
cust_id=1
result=rentals.merge(movies, on="movie_id")
result[result["customer_id"]==cust_id][["customer_id","title","genre"]]

,customer_id,title,genre
3,1,Shrek,Animation
15,1,Joker,Drama
16,1,Joker,Drama
24,1,Frozen,Animation


In [4]:
top_movies=rentals.groupby("movie_id").size().reset_index(name="count")
top_movies=top_movies.merge(movies, on="movie_id")
top_movies=top_movies.sort_values("count",ascending=False)
top_movies.head()

,movie_id,count,title,genre,release_year,rating
3,4,6,Joker,Drama,2019,5
0,1,4,Inception,Sci-Fi,2010,5
8,10,4,Gladiator,Action,2000,5
1,2,3,Titanic,Romance,1997,5
4,5,2,Frozen,Animation,2013,4


In [6]:
#Compute the total revenue per genre (join rentals with movies, group by genre, sum price).
rev=rentals.merge(movies,on="movie_id")
rev=rev.groupby("genre")["price"].sum()
rev

genre
Action       55
Animation    51
Drama        56
Romance      40
Sci-Fi       64
Name: price, dtype: int32

In [7]:
merged=rentals.merge(movies, on="movie_id")
bad_customers=merged[merged["rating"]<3]["customer_id"].unique()
customers[~customers["customer_id"].isin(bad_customers)]

,customer_id,name,email,city,signup_date
0,1,Ali,ali@mail.com,Baku,2023-01-01
1,2,Nigar,nigar@mail.com,Ganja,2023-02-10
2,3,Murad,murad@mail.com,Sumgait,2023-03-15
3,4,Aysel,aysel@mail.com,Baku,2023-04-20
4,5,Rashad,rashad@mail.com,Baku,2023-05-05
5,6,Leyla,leyla@mail.com,Ganja,2023-06-01
6,7,Tural,tural@mail.com,Sumgait,2023-06-10
7,8,Sabina,sabina@mail.com,Baku,2023-07-01


In [8]:
dernormalized=rentals.merge(customers,on="customer_id").merge(movies,on="movie_id")
dernormalized.head()

,rental_id,customer_id,movie_id,rental_date,return_date,price,name,email,city,signup_date,title,genre,release_year,rating
0,1,5,2,2023-08-01,2023-08-05,12,Rashad,rashad@mail.com,Baku,2023-05-05,Titanic,Romance,1997,5
1,2,8,6,2023-08-02,2023-08-06,8,Sabina,sabina@mail.com,Baku,2023-07-01,Matrix,Sci-Fi,1999,5
2,3,6,10,2023-08-03,2023-08-07,19,Leyla,leyla@mail.com,Ganja,2023-06-01,Gladiator,Action,2000,5
3,4,1,9,2023-08-04,2023-08-08,16,Ali,ali@mail.com,Baku,2023-01-01,Shrek,Animation,2001,3
4,5,4,10,2023-08-05,2023-08-09,7,Aysel,aysel@mail.com,Baku,2023-04-20,Gladiator,Action,2000,5


Redundancy in the denormalized table:

Customer information (name, email, city) is repeated for every rental that the customer made.

Movie information (title, genre, rating) is repeated for every rental of that movie.

This leads to storage inefficiency and data duplication.

Why normalized tables are preferable:

In a normalized schema, customer data and movie data are stored only once in separate tables.

The rentals table references customers and movies via their IDs, avoiding repetition.

Benefits:

Updates are simpler and less error-prone (e.g., changing a customer's email requires updating only one row).

Data consistency is maintained.

Less storage space is used.

In [10]:
documents = []

for _, cust in customers.iterrows():
    cust_rentals = rentals[rentals["customer_id"] == cust["customer_id"]]
    
    rental_list = []
    for _, r in cust_rentals.iterrows():
        movie = movies[movies["movie_id"] == r["movie_id"]].iloc[0]
        
        rental_list.append({
            "movie_title": movie["title"],
            "genre": movie["genre"],
            "rating": movie["rating"],
            "price": r["price"],
            "rental_date": r["rental_date"],
            "return_date": r["return_date"]
        })
    
    documents.append({
        "customer_id": cust["customer_id"],
        "name": cust["name"],
        "email": cust["email"],
        "city": cust["city"],
        "signup_date": cust["signup_date"],
        "rentals": rental_list
    })

documents[0]

{'customer_id': 1,
 'name': 'Ali',
 'email': 'ali@mail.com',
 'city': 'Baku',
 'signup_date': '2023-01-01',
 'rentals': [{'movie_title': 'Shrek',
   'genre': 'Animation',
   'rating': np.int64(3),
   'price': 16,
   'rental_date': Timestamp('2023-08-04 00:00:00'),
   'return_date': Timestamp('2023-08-08 00:00:00')},
  {'movie_title': 'Joker',
   'genre': 'Drama',
   'rating': np.int64(5),
   'price': 9,
   'rental_date': Timestamp('2023-08-16 00:00:00'),
   'return_date': Timestamp('2023-08-20 00:00:00')},
  {'movie_title': 'Joker',
   'genre': 'Drama',
   'rating': np.int64(5),
   'price': 6,
   'rental_date': Timestamp('2023-08-17 00:00:00'),
   'return_date': Timestamp('2023-08-21 00:00:00')},
  {'movie_title': 'Frozen',
   'genre': 'Animation',
   'rating': np.int64(4),
   'price': 12,
   'rental_date': Timestamp('2023-08-25 00:00:00'),
   'return_date': Timestamp('2023-08-29 00:00:00')}]}

In [12]:
cust_id=1
movies_rented=[]
for c in documents:
    if c["customer_id"]==cust_id:
        for r in c["rentals"]:
            movies_rented.append(r["movie_title"])
movies_rented    

['Shrek', 'Joker', 'Joker', 'Frozen']

In [13]:
from collections import Counter

counter = Counter()

for c in documents:
    for r in c["rentals"]:
        counter[r["movie_title"]] += 1

counter.most_common(5)

[('Joker', 6),
 ('Inception', 4),
 ('Gladiator', 4),
 ('Titanic', 3),
 ('Shrek', 2)]

In [14]:
rev={}

for c in documents:
    for r in c["rentals"]:
        rev[r["genre"]] = rev.get(r["genre"], 0) + r["price"]
rev

{'Animation': 51, 'Drama': 56, 'Sci-Fi': 64, 'Action': 55, 'Romance': 40}

In [15]:
good_customers=[]
for c in documents:
    if all(r["rating"]>3 for r in c["rentals"]):
        good_customers.append(c["name"])
good_customers

['Nigar', 'Murad', 'Aysel', 'Rashad', 'Leyla', 'Tural']

Easier in document model:
Customer-specific queries, because all rentals are nested inside the customer document

No joins needed to get all rentals of a customer

Harder in document model:
Aggregations (e.g., total revenue, top movies) require explicit loops

Relational model with groupby and merge handles this more efficiently

Advantage of document model:
Read-heavy, customer-centric queries benefit from locality

Updates to nested data are easy if you only care about one customer

Flexible schema allows different fields for different documents

In [16]:
nodes = {}
edges = []

for _, c in customers.iterrows():
    nodes[f"c{c.customer_id}"] = {"type": "customer", "name": c.name}

for _, m in movies.iterrows():
    nodes[f"m{m.movie_id}"] = {"type": "movie", "title": m.title}

for g in movies["genre"].unique():
    nodes[g] = {"type": "genre", "name": g}

for _, r in rentals.iterrows():
    edges.append((f"c{r.customer_id}", "rented", f"m{r.movie_id}"))

for _, m in movies.iterrows():
    edges.append((f"m{m.movie_id}", "belongs_to", m.genre))

We represent data as a graph:

Nodes = customers, movies, genres

Edges = relationships between them

Relationships:

rented: customer → movie

belongs_to: movie → genre

This structure allows easy traversal of relationships

In [30]:
def movies_of_customer(cid):
    movies_list = []
    
    for src, rel, dst in edges:
        if src == f"c{cid}" and rel == "rented":
            movies_list.append(nodes[dst]["title"])
    
    return movies_list


movies_of_customer(2)

['Joker', 'Inception']

In [18]:
#"Which genres does customer X prefer?" two hops: customer → movies → genres, then count.
from collections import Counter

def preferred_genres(cid):
    genres = []
    
    for src, rel, movie in edges:
        if src == f"c{cid}" and rel == "rented":
            for s, r, g in edges:
                if s == movie and r == "belongs_to":
                    genres.append(g)
    
    return Counter(genres)

preferred_genres(1)

Counter({'Animation': 2, 'Drama': 2})

In [19]:
#"Which customers share 
#a genre preference with customer X?" — three hops: customer X → movies → genres → other customers who rented movies in the same genre.
def similar_customers(cid):
    genres = set(preferred_genres(cid).keys())
    result = set()
    
    for edge in edges:
        if edge[1] == "rented":
            other_customer = edge[0]
            movie_id = edge[2]
            
            for e in edges:
                if e[0] == movie_id and e[1] == "belongs_to":
                    genre = e[2]
                    
                    if genre in genres and other_customer != f"c{cid}":
                        result.add(other_customer)
    
    return result 
similar_customers(1)

{'c2', 'c4', 'c5', 'c8'}

In [20]:
#"Recommend movies for customer X" — find movies in their preferred genres that they have not yet rented.
def recommend_movies(cid):
    watched = set()
    
    for edge in edges:
        if edge[0] == f"c{cid}" and edge[1] == "rented":
            watched.add(edge[2])
    
    genres = set(preferred_genres(cid).keys())
    recommendations = []
    
    for edge in edges:
        if edge[1] == "belongs_to":
            movie_id = edge[0]
            genre = edge[2]
            
            if genre in genres and movie_id not in watched:
                recommendations.append(nodes[movie_id]["title"])
    
    return recommendations

recommend_movies(3)

['Inception', 'Matrix', 'Batman', 'Gladiator']


['Inception', 'Matrix', 'Batman', 'Gladiator']
In the graph model:

We do not use joins

We simply traverse edges (relationships)

For example:

1-hop → customer → movie

2-hop → customer → movie → genre

3-hop → customer → movie → genre → other customers

👉 Because of this:

The code is simpler

The logic is clearer and more intuitive

The graph model is especially powerful for recommendation systems and relationship-based queries

In [21]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 5000

transactions = pd.DataFrame({
    "transaction_id": range(1, n+1),
    "customer_id": np.random.randint(1, 100, n),
    "product_id": np.random.randint(1, 50, n),
    "amount": np.round(np.random.uniform(5, 500, n), 2),
    "payment_method": np.random.choice(["card", "cash", "paypal"], n),
    "timestamp": pd.date_range(start="2024-01-01", periods=n, freq="h"),
    "status": np.random.choice(["completed", "pending", "refunded"], n)
})
transactions.head()

,transaction_id,customer_id,product_id,amount,payment_method,timestamp,status
0,1,52,39,256.70,cash,2024-01-01 00:00:00,pending
1,2,93,47,120.29,paypal,2024-01-01 01:00:00,pending
2,3,15,19,423.44,paypal,2024-01-01 02:00:00,refunded
3,4,72,15,364.95,paypal,2024-01-01 03:00:00,refunded
4,5,61,28,490.48,card,2024-01-01 04:00:00,pending


In [22]:
#Step 2
#Look up a single transaction by its ID (simulate a point query).

#Insert a new transaction (append a row).

#Update the status of a transaction from "pending" to "completed."

#Check that the transaction is valid: the amount must be positive and the status must be one of the allowed values (simulate a consistency check).

transactions[transactions["transaction_id"]==100]

,transaction_id,customer_id,product_id,amount,payment_method,timestamp,status
99,100,89,15,22.99,cash,2024-01-05 03:00:00,pending


In [23]:

#Insert a new transaction (append a row).
new_row = pd.DataFrame([{
    "transaction_id": 5001,
    "customer_id": 10,
    "product_id": 5,
    "amount": 120.0,
    "payment_method": "card",
    "timestamp": pd.Timestamp.now(),
    "status": "pending"
}])

transactions=pd.concat([transactions,new_row],ignore_index=True)

In [24]:
##Update the status of a transaction from "pending" to "completed."
transactions.loc[transactions["transaction_id"]==1,"status"]="pending"
transactions[transactions["transaction_id"]==1]

,transaction_id,customer_id,product_id,amount,payment_method,timestamp,status
0,1,52,39,256.7,cash,2024-01-01,pending


In [25]:
#Check that the transaction is valid: the amount must be positive and the status must be one of the allowed values (simulate a consistency check).
valid=transactions[ (transactions["amount"]>0) & (transactions["status"].isin(['pending', 'refunded', 'completed']))]
len(valid)==len(transactions)

True

In [26]:

#Step3 Compute total revenue by payment method.
total_rev=transactions.groupby("payment_method")["amount"].sum()
total_rev

payment_method
card      418253.01
cash      415186.51
paypal    415797.42
Name: amount, dtype: float64

In [27]:
#Find the average transaction amount by month.
transactions["month"]=transactions["timestamp"].dt.to_period("M")
avg_amount=transactions.groupby("month")["amount"].mean()
avg_amount

month
2024-01    250.307312
2024-02    239.551451
2024-03    246.210484
2024-04    251.381278
2024-05    248.137634
2024-06    257.360986
2024-07    256.441551
2026-03    120.000000
Freq: M, Name: amount, dtype: float64

In [28]:
#Identify the top 10 customers by total spending
transactions.groupby("customer_id")["amount"].sum().nlargest(10)

customer_id
47    17584.80
62    17138.14
33    16937.96
87    16889.69
8     16617.38
21    16220.52
15    15992.04
51    15777.26
35    15605.46
19    15141.65
Name: amount, dtype: float64

In [29]:
#Compute the refund rate (percentage of refunded transactions)
refund_rate=(transactions["status"]=="refunded").mean()*100
refund_rate

np.float64(32.63347330533893)

OLTP (Online Transaction Processing) systems are designed for fast, real-time operations such as inserts, updates, and single-record lookups. These operations typically access all fields of a single row (e.g., one transaction). Row-major storage keeps all attributes of a record together in memory, making these operations very efficient.

On the other hand, OLAP (Online Analytical Processing) systems are optimized for analytical queries that process large amounts of data, often focusing on a few columns (e.g., calculating average amount or total revenue). Column-major storage stores data column by column, allowing systems to scan only the relevant columns and apply vectorized operations efficiently.

A hospital patient record system where doctors need to quickly look up a patient's complete medical history.- Relational Model + OLTP A hospital system requires highly structured and consistent data (patients, doctors, treatments, prescriptions). The relational model ensures data integrity through constraints and relationships. Since doctors need fast lookups and frequent updates (e.g., adding diagnoses, updating records), this is a classic OLTP workload. Reliability and accuracy are critical, making relational databases the best choice

A social network that needs to suggest "friends of friends" connections. -Graph Model + OLTP Social networks are built on relationships (users connected to other users). Queries like “friends of friends” require multi-hop traversal, which is exactly what graph databases excel at. These systems also need to respond quickly to user interactions (adding friends, likes), which fits OLTP workloads. Graph models make these relationship queries much simpler and faster.

A financial reporting system that computes monthly revenue breakdowns across product categories and regions.-Relational Model (Data Warehouse) + OLAP Financial reporting involves aggregations like monthly revenue, regional breakdowns, and trends over time. These are analytical queries over large datasets, making OLAP the right paradigm. A relational data warehouse (often column-based) is ideal because it supports fast aggregations and structured reporting. Performance for reads and summaries is more important than frequent updates

An IoT platform that receives sensor readings from 10,000 devices and needs to detect anomalies in real time.-Document Model + OLTP IoT systems receive high volumes of semi-structured data from many devices. Each device may send slightly different data formats, making the document model flexible and suitable. Since data is continuously inserted and processed in real time (e.g., anomaly detection), this is an OLTP-style workload. Document databases handle scalability and schema flexibility well.

A content management system where each article has a different set of metadata fields.-Document Model + OLTP In a CMS, each article can have different metadata (tags, categories, author info, custom fields). The document model allows storing flexible, nested structures without a fixed schema. Since users frequently create, edit, and retrieve articles, this is an OLTP workload. Document databases make it easy to evolve the structure without complex schema migrations.